# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   null|     BE|   null|    null|      1|  null|   269|  6|    69| null|       1|    null|    0.0|    null|    null|    null|    null|    null|    null|    null|
|3070802| 1963| 1096|   null|     US|     TX|    null|      1|  null|     2|  6|    63| null|       0|    null|   null|    null|    null|    null|    null|    null|    null|    null|
|3070803| 1963| 1096|   null|     US|     IL|    null|      1|  null|     2|  6|    6

In [68]:
# Remove all patents without a POSTATE citation
patents = patents.filter(patents.POSTATE.isNotNull())
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070802| 1963| 1096|   null|     US|     TX|    null|      1|  null|     2|  6|    63| null|       0|    null|   null|    null|    null|    null|    null|    null|    null|    null|
|3070803| 1963| 1096|   null|     US|     IL|    null|      1|  null|     2|  6|    63| null|       9|    null| 0.3704|    null|    null|    null|    null|    null|    null|    null|
|3070804| 1963| 1096|   null|     US|     OH|    null|      1|  null|     2|  6|    6

In [85]:
# Create aliases for the tables
p1 = patents.alias('p1')
p2 = patents.select("PATENT","POSTATE")
p2 = p2.withColumnRenamed("PATENT","PATENT2") \
       .withColumnRenamed("POSTATE","POSTATE2")
c = citations.alias('c')

# Join together tables based on if patents were from the same state
joined = c.join(p1, c.CITING == p1.PATENT) \
          .join(p2, c.CITED == p2.PATENT2) \
          .filter(p1.POSTATE == p2.POSTATE2)
joined.select("CITING","PATENT","POSTATE","CITED","PATENT2","POSTATE2").show(5)

+-------+-------+-------+-------+-------+--------+
| CITING| PATENT|POSTATE|  CITED|PATENT2|POSTATE2|
+-------+-------+-------+-------+-------+--------+
|4458718|4458718|     CA|3076476|3076476|      CA|
|4664151|4664151|     CA|3076476|3076476|      CA|
|4780050|4780050|     IL|3078806|3078806|      IL|
|4275169|4275169|     NJ|3080256|3080256|      NJ|
|3960588|3960588|     NJ|3080256|3080256|      NJ|
+-------+-------+-------+-------+-------+--------+
only showing top 5 rows



In [86]:
# Cache the df for future use
joined.cache()

DataFrame[CITING: int, CITED: int, PATENT: int, GYEAR: int, GDATE: int, APPYEAR: int, COUNTRY: string, POSTATE: string, ASSIGNEE: int, ASSCODE: int, CLAIMS: int, NCLASS: int, CAT: int, SUBCAT: int, CMADE: int, CRECEIVE: int, RATIOCIT: double, GENERAL: double, ORIGINAL: double, FWDAPLAG: double, BCKGTLAG: double, SELFCTUB: double, SELFCTLB: double, SECDUPBD: double, SECDLWBD: double, PATENT2: int, POSTATE2: string]

In [98]:
# Generated co-state citation counts
counts = joined.groupBy("CITING").count().orderBy("count",ascending=False) # Used orderBy to compare counts to solution
counts = counts.withColumnRenamed("count","SAME_STATE")
counts.show(10)

+-------+----------+
| CITING|SAME_STATE|
+-------+----------+
|5959466|       125|
|5983822|       103|
|6008204|       100|
|5952345|        98|
|5958954|        96|
|5998655|        96|
|5936426|        94|
|5980517|        90|
|5925042|        90|
|5951547|        90|
+-------+----------+
only showing top 10 rows



In [100]:
# Append the counts to the patents data frame for display
results = patents.join(counts, patents.PATENT == counts.CITING)
results.orderBy("SAME_STATE",ascending=False).show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD| CITING|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  null|   326|  4|    46|  159|       0|     1.0|   null|  0.6186|    null|  4.8868|  0.0455|   0.044|    null|    null|5959466|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  null|   114|  5|    55|  200|       0|   0.995|   null|  0.7201|    null|   12.45|     0.0|     0.0|    null|    null|5983822|  